In [ ]:
import geopandas as gpd
import pandas as pd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/GE/step-1'
output_step2_path='../../Data/output/GE/step-2'
output_step3_path='../../Data/output/GE/step-3'

#Read the files
index_walkability = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index_walkability = index_walkability.to_crs(operation_crs)

zones_girec = gpd.read_file(f'{input_file_path}/network_agreg/GEO_GIREC-SHP/GEO_GIREC.shp')
zones_girec = zones_girec.to_crs(operation_crs)

agglo_carreau = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_CARREAU_200-SHP/AGGLO_CARREAU_200.shp')
agglo_carreau = agglo_carreau.to_crs(operation_crs)

zones_communes = gpd.read_file(f'{input_file_path}/network_agreg/CAD_COMMUNE-SHP/CAD_COMMUNE.shp')
zones_communes = zones_communes.to_crs(operation_crs)

zones_communes_GE_fusionnee = gpd.read_file(f'{input_file_path}/network_agreg/CAD_COMMUNE-SHP/CAD_COMMUNES_GE_fusionnee.shp')
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.to_crs(operation_crs)


**GIREC**

In [ ]:
# Spatial join
segments_girec = gpd.sjoin(index_walkability, zones_girec, how="inner", predicate="within")

# Columns to aggregate
cols = index_walkability.columns.to_list()

def weighted_mean(df, cols, weight_col):
    return (df[cols].multiply(df[weight_col], axis=0).sum() / df[weight_col].sum())

cols_to_agg = cols[3:]  # tes colonnes d'indicateurs

# Calcul pondéré
girec_stats = (
    segments_girec
        .groupby("OBJECTID")
        .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
        .reset_index()
)

# Merge back with zones_mmt polygons
zones_girec = zones_girec.merge(girec_stats, on="OBJECTID", how="left")
zones_girec

#Drop nan values 
zones_girec = zones_girec.dropna(subset=["walk_index"])

In [ ]:
zones_girec.head()

**Carreau 200**

In [ ]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_carreau = gpd.sjoin(index_walkability, agglo_carreau, how="inner", predicate="within")

# Aggregate by mean
carreau_stats = (
    segments_carreau
    .groupby("GRID_ID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
agglo_carreau = agglo_carreau.merge(carreau_stats, on="GRID_ID", how="left")

# Drop rows with missing values (optional)
agglo_carreau = agglo_carreau.dropna(subset=["walk_index"])

In [ ]:
segments_carreau.head()

**[Communes](https://sitg.ge.ch/donnees/cad-commune)**

In [ ]:
zones_communes.head()

In [ ]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_communes = gpd.sjoin(index_walkability, zones_communes, how="inner", predicate="within")

# Aggregate by mean
communes_stats = (
    segments_communes
    .groupby("OBJECTID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
zones_communes = zones_communes.merge(communes_stats, on="OBJECTID", how="left")

# Drop rows with missing values (optional)
zones_communes = zones_communes.dropna(subset=["walk_index"])

In [ ]:
zones_communes.head()

*Communes avec Genève fusionnée (1 seul polygone au lieu de 4 différents)*

In [ ]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_communes_GE_fusionnee = gpd.sjoin(index_walkability, zones_communes_GE_fusionnee, how="inner", predicate="within")

# Aggregate by mean
communes_GE_fusionnee_stats = (
    segments_communes_GE_fusionnee
    .groupby("OBJECTID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.merge(communes_GE_fusionnee_stats, on="OBJECTID", how="left")

# Drop rows with missing values (optional)
zones_communes_GE_fusionnee = zones_communes_GE_fusionnee.dropna(subset=["walk_index"])

In [ ]:
#save the file 
zones_girec.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_girec.gpkg"), driver="GPKG")
zones_girec.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_girec.parquet')

agglo_carreau.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_carreau200.gpkg"), driver="GPKG")
agglo_carreau.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_carreau200.parquet')

zones_communes.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_communes.gpkg"), driver="GPKG")
zones_communes.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_communes.parquet')

zones_communes_GE_fusionnee.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_communes_GE_fusionnee.gpkg"), driver="GPKG")
zones_communes_GE_fusionnee.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_communes_GE_fusionnee.parquet')

**[POPULATION 2025](https://statistique.ge.ch/atlas/index.php#c=indicator&i=population.pop_tot&s=2025&t=A01&view=map3)**

In [ ]:
pop_communes = pd.read_csv(f'{input_file_path}/STAT/POPULATION/pop_2025.csv', sep=";", header=2)
pop_communes = pop_communes.rename(columns={"Population résidante 2025":"pop_2025"})

colonnes_pop = ['pop_2025']

pop_communes[colonnes_pop] = pop_communes[colonnes_pop].apply(pd.to_numeric, errors='coerce')


In [ ]:
pop_communes.head()

In [ ]:
zones_communes_GE_fusionnee = (zones_communes_GE_fusionnee.merge(
    pop_communes[["Code"] + colonnes_pop],
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
).drop(columns="Code"))

In [ ]:
zones_communes_GE_fusionnee.head()

**[FREQUENCE CRIMINALITE GENEVE 2024](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)**

In [ ]:
criminalite_communes = pd.read_csv(f'{input_file_path}/STAT/CRIMINALITE/CRIMINALITE_GE_2024.csv', sep=";", header=2)

criminalite_communes = criminalite_communes.rename(columns={"Loi sur les stupéfiants (LStup) : fréquence d'infractions 2024":"freq_LStup_infra","Code pénal (CP) : fréquence d'infractions 2024":"freq_CP_infra", "Loi sur les étrangers et l’intégration (LEI) : fréquence d'infractions 2024":"freq_LEI_infra"})

#convert string to float
colonnes_crim = ['freq_LStup_infra', 'freq_CP_infra', 'freq_LEI_infra']

criminalite_communes[colonnes_crim] = criminalite_communes[colonnes_crim].apply(pd.to_numeric, errors='coerce')

criminalite_communes['freq_crim_mean'] = criminalite_communes[colonnes_crim].mean(axis=1)

#print(criminalite_communes[colonnes].dtypes)

In [ ]:
criminalite_communes

In [ ]:
zones_communes_GE_fusionnee = (zones_communes_GE_fusionnee.merge(
    criminalite_communes[["Code"]+ colonnes_crim + ["freq_crim_mean"]],
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
).drop(columns="Code"))

In [ ]:
zones_communes_GE_fusionnee.head()

**[TAUX MOTORISATION CANTON GENEVE](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)**

In [ ]:
taux_motorisation_communes = pd.read_csv(f'{input_file_path}/STAT/TAUX_MOTORISATION/taux_motorisation_2024.csv', sep=";", header=2)

In [ ]:
taux_motorisation_communes.head()

In [ ]:
taux_motorisation_communes = taux_motorisation_communes.rename(columns={"Taux de motorisation 2024": "freq_voitures"})

#convert string to float
colonnes_motor = ["freq_voitures"]

taux_motorisation_communes[colonnes_motor] = taux_motorisation_communes[colonnes_motor].apply(pd.to_numeric, errors='coerce')

In [ ]:
taux_motorisation_communes.head()

In [ ]:
zones_communes_GE_fusionnee = (zones_communes_GE_fusionnee.merge(
    taux_motorisation_communes[["Code"] + colonnes_motor],
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
).drop(columns="Code"))

In [ ]:
zones_communes_GE_fusionnee.head()

**[CHOMAGE](https://statistique.ge.ch/atlas/index.php#c=indicator&view=map3)**

In [ ]:
chomage_communes = pd.read_csv(f'{input_file_path}/STAT/CHOMAGE/chomage_2025.csv', sep=";", header=2)

In [ ]:
chomage_communes.head()

In [ ]:
chomage_communes = chomage_communes.rename(columns={"Chômeurs inscrits 2025": "nb_chomage"})

#convert string to float
colonnes_chomage = ["nb_chomage"]

chomage_communes[colonnes_chomage] = chomage_communes[colonnes_chomage].apply(pd.to_numeric, errors='coerce')

In [ ]:
chomage_communes.head()

In [ ]:
zones_communes_GE_fusionnee = (zones_communes_GE_fusionnee.merge(
    chomage_communes[["Code"] + colonnes_chomage],
    left_on="NO_COM_FED",
    right_on="Code",
    how="left" 
).drop(columns="Code"))

In [ ]:
zones_communes_GE_fusionnee.head()

In [ ]:
zones_communes_GE_fusionnee["freq_chomage"] = (
    zones_communes_GE_fusionnee["nb_chomage"] /
    zones_communes_GE_fusionnee["pop_2025"]
).replace([float("inf")], pd.NA) * 1000

zones_communes_GE_fusionnee.drop(columns="nb_chomage", inplace=True)

zones_communes_GE_fusionnee.head()

**EXPORT**

In [ ]:
zones_communes_GE_fusionnee.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_communes_GE_fusionnee_STAT.gpkg"), driver="GPKG")
zones_communes_GE_fusionnee.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_communes_GE_fusionnee_STAT.parquet')

**CORRELATION**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

colonnes_stat = ['walk_index', 
                 'freq_LStup_infra', 
                 'freq_CP_infra', 
                 'freq_LEI_infra', 
                 #'freq_crim_mean', 
                 'freq_voitures',
                 'freq_chomage']

zones_communes_GE_fusionnee_corr = zones_communes_GE_fusionnee[colonnes_stat].corr(method='pearson')
#print(zones_communes_corr)

sns.heatmap(zones_communes_GE_fusionnee_corr, annot=True, cmap='coolwarm')
plt.title("Pearson Correlation Heatmap")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for var in colonnes_stat:
    sns.regplot(data=zones_communes_GE_fusionnee, x=var, y='walk_index')
    plt.title(f"Relation entre {var} et walk_index")
    plt.show()

In [ ]:
import statsmodels.api as sm

variables = [ 'freq_LStup_infra', 
                 'freq_CP_infra', 
                 'freq_LEI_infra', 
                 #'freq_crim_mean', 
                 'freq_voitures',
                 'freq_chomage']

data_reg = zones_communes_GE_fusionnee[variables + ["walk_index"]].dropna()

communes_Nan = zones_communes_GE_fusionnee[['COMMUNE'] + colonnes_stat].loc[zones_communes_GE_fusionnee[colonnes_stat].isna().any(axis=1)]
communes_Nan.head()

In [ ]:
X = data_reg[variables]
y = data_reg['walk_index']

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

variables = [
    'freq_LStup_infra',
    'freq_CP_infra',
    'freq_LEI_infra',
    #'freq_crim_mean',
    'freq_voitures',
    'freq_chomage'
]

data_rf = zones_communes_GE_fusionnee[variables + ['walk_index']].dropna()

X = data_rf[variables]
y = data_rf['walk_index']

model = RandomForestRegressor(
    n_estimators=1000,
    random_state=42
)

model.fit(X, y)

importance = pd.Series(
    model.feature_importances_,
    index=variables
).sort_values(ascending=False)

print(importance)

In [ ]:
import matplotlib.pyplot as plt

importance.sort_values().plot.barh()

plt.xlabel("Importance")
plt.title("Importance des variables expliquant le walk_index")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

variables = [
    'walk_index',
    'freq_LStup_infra',
    'freq_CP_infra',
    'freq_LEI_infra',
    #'freq_crim_mean',
    'freq_voitures',
    'freq_chomage'
]

data_plot = zones_communes_GE_fusionnee[variables].dropna()

In [ ]:
sns.pairplot(
    data_plot,
    kind="reg",
    diag_kind="kde"
)

plt.show()

In [ ]:
zones_communes_GE_fusionnee["type_commune"] = "Autres"
zones_communes_GE_fusionnee.loc[
    zones_communes_GE_fusionnee["COMMUNE"] == "Genève",
    "type_commune"
] = "Genève"

In [ ]:
sns.pairplot(
    zones_communes_GE_fusionnee[variables + ["type_commune"]].dropna(),
    hue="type_commune",
    kind="reg",
    diag_kind="kde"
)

plt.show()